# Partie 3 — Construction du panel sentiment / marché

**Ce que fait ce notebook.** Il transforme les **messages StockTwits scorés par FinBERT** en un tableau de
**2 740 lignes × 76 colonnes** : une ligne par couple (action, jour de bourse).

C'est l'étape que la partie 4 suppose déjà faite. Tant qu'on ne l'a pas comprise, aucune conclusion de la
partie 4 n'est défendable — parce que **c'est ici que se jouent les choix qui déterminent tout le reste** :
l'heure de référence, le rattachement des messages aux jours de bourse, et le découpage en fenêtres.

---

## Le point de départ : ton fichier

Corpus **StockTwits** récupéré sur Kaggle — messages *scrapés* (pas l'API), pour cinq titres, de janvier
2020 à mars 2022. Après ton nettoyage et l'inférence FinBERT, il a **huit colonnes** :

| Colonne | Contenu | Rôle ici |
|---|---|---|
| `Tweet` | le message brut | archive — non utilisé |
| `Texte_Nettoye` | le message nettoyé, celui donné à FinBERT | archive — non utilisé |
| `Ticker` | AAPL, AMZN, FB/META, NVDA, TSLA | clé de regroupement |
| `Jour` | la date du message | **rattachement au jour de bourse** |
| `Heure_decimale` | l'heure en décimal (0,616667 = 00h37) | **découpage en fenêtres** |
| `FinBERT_Positive` | p(positif) | → `pos` |
| `FinBERT_Negative` | p(négatif) | → `neg` |
| `FinBERT_Neutral` | p(neutre) | contrôle de qualité |

**L'horodatage est donc déjà découpé en deux colonnes** — c'est plus simple, mais cela crée un piège
particulier qu'on traite au §2.

---

## Le problème, en une phrase

Les messages sont **continus** (une heure à la minute près), les prix sont **discrets** (une ligne par jour
de bourse, avec des trous les week-ends et les jours fériés). Il faut donc décider, pour chaque message,
**à quel jour de bourse il appartient** et **dans quelle phase de la journée il tombe**.

```
   Messages (Kaggle + FinBERT)                          Panel
   ┌──────────────────────────┐                    ┌──────────────┐
   │ Ticker                   │  ── heure de ───►  │ Date         │
   │ Jour                     │     référence      │ Ticker       │
   │ Heure_decimale           │  ── calendrier ─►  │ n_overnight  │
   │ FinBERT_Positive         │  ── fenêtres ───►  │ mu_overnight │
   │ FinBERT_Negative         │  ── agrégation ─►  │ ...          │
   │ FinBERT_Neutral          │  ── dérivées ───►  │ y_gap        │
   └──────────────────────────┘                    └──────────────┘
```

---

## Les trois décisions qui font tout

| Décision | Le piège | Ce qu'on fait |
|---|---|---|
| **Heure de référence** | `Heure_decimale` est-elle en heure de New York ou en UTC ? Le fichier ne le dit pas — et 5 h d'écart déplacent la moitié des messages de fenêtre | on le **détermine par les données** (§2) |
| **Jour de bourse** | un message du samedi appartient au lundi ; un message du 25 décembre au 26 | calendrier déduit **des prix eux-mêmes**, ce qui gère les fériés |
| **Fenêtre** | la fenêtre nocturne du lundi couvre tout le week-end, pas seulement la nuit | définition explicite et vérifiable |

---

## Plan

| § | Étape |
|---|---|
| **1** | Charger les messages et reconstituer un horodatage |
| **1bis** | Les étiquettes Bullish / Bearish — une occasion à ne pas manquer |
| **2** | Heure de New York ou UTC ? — la question à trancher |
| **3** | Le calendrier de bourse, déduit des prix |
| **4** | Affecter chaque message à un jour de bourse et à une fenêtre |
| **5** | Agréger : les 7 statistiques par (action, jour, fenêtre) |
| **6** | Passer au format large — une ligne par action-jour |
| **7** | Les variables de marché : `gap`, `ret_oc`, `ret_cc`, `vol_20d` |
| **8** | Fusionner texte et marché |
| **9** | Les variables d'attention : `nlog`, `nabn`, `disp` |
| **10** | Les dérivées nocturnes : `dmu_night`, `ma3`, `z20` |
| **11** | Les variables décalées et les cibles |
| **12** | Contrôles qualité et export |

---
# §1 — Charger les messages et reconstituer un horodatage

### Ce que produit FinBERT

Pour chaque message, FinBERT renvoie **trois probabilités qui somment à 1** :

```
FinBERT_Positive  +  FinBERT_Neutral  +  FinBERT_Negative  =  1
```

On en tire le **score** du message :

```
score = FinBERT_Positive − FinBERT_Negative        (entre −1 et +1)
```

Regarde tes cinq premières lignes, elles illustrent bien les trois cas :

| Message | pos | neg | neu | score | lecture |
|---|---|---|---|---|---|
| « Happy New Year amazing winning AAPL Bull… » | 1,000 | 0,000 | 0,000 | **+1,00** | franchement haussier |
| « my dad ended this year by selling half his $aapl… » | 0,000 | 1,000 | 0,000 | **−1,00** | franchement baissier |
| « Happy New Year :) $AAPL $TSLA $AMZN $SPY » | 0,000 | 0,000 | 1,000 | **0,00** | aucune information |

Le troisième cas est le plus fréquent, et c'est important pour la suite : un score de 0 ne veut pas dire
« la foule hésite », il veut dire « FinBERT n'y voit aucun contenu financier ». On garde donc `pos` et
`neg` séparément, pour pouvoir distinguer *neutre* de *partagé*.

### Reconstituer un horodatage unique

`Jour` et `Heure_decimale` sont deux moitiés du même objet. On les recolle en un seul horodatage, ce qui
rendra tout le reste plus simple et permettra de gérer la conversion d'heure au §2 :

```
horodatage = Jour + Heure_decimale heures
```

In [ ]:
# ============================================================================
#  CONFIGURATION — la seule cellule à adapter
# ============================================================================
PROJET = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project"

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = os.path.join(PROJET, "data", "processed")
DOCS = os.path.join(PROJET, "docs")
os.makedirs(DOCS, exist_ok=True)
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 80)
plt.style.use("seaborn-v0_8-whitegrid")

MESSAGES = os.path.join(DATA, "STOCKTWITS_2020_2022_FINBERT.pkl")   # .pkl / .csv / .parquet
FINANCE  = os.path.join(DATA, "FINANCE_2020_2022.csv")
SORTIE   = os.path.join(DATA, "PANEL_SENTIMENT_WINDOWS_2020_2022_v2.csv")

# Les colonnes de TON fichier
C_JOUR, C_HEURE, C_TICKER = "Jour", "Heure_decimale", "Ticker"
C_POS, C_NEG, C_NEU = "FinBERT_Positive", "FinBERT_Negative", "FinBERT_Neutral"

TICKERS = ["AAPL", "AMZN", "META", "NVDA", "TSLA"]
DEBUT, FIN = "2020-01-02", "2022-03-04"
TZ_MARCHE = "America/New_York"


def charger(chemin):
    ext = os.path.splitext(chemin)[1].lower()
    if ext == ".pkl":     return pd.read_pickle(chemin)
    if ext == ".parquet": return pd.read_parquet(chemin)
    return pd.read_csv(chemin)


msg = charger(MESSAGES)
print(f"{len(msg):,} messages chargés")
print(f"Colonnes : {list(msg.columns)}\n")

attendues = [C_JOUR, C_HEURE, C_TICKER, C_POS, C_NEG]
absentes = [c_ for c_ in attendues if c_ not in msg.columns]
if absentes:
    raise KeyError(f"Colonnes introuvables : {absentes}\n"
                   f"Corrige les constantes C_JOUR / C_HEURE / ... ci-dessus.\n"
                   f"Colonnes disponibles : {list(msg.columns)}")
print("Toutes les colonnes attendues sont présentes.")

In [ ]:
# --- Table de travail : 5 colonnes, rien de plus ---------------------------
m = pd.DataFrame({
    "Ticker": msg[C_TICKER].astype(str).str.upper().str.lstrip("$").str.strip(),
    "jour_src": pd.to_datetime(msg[C_JOUR], errors="coerce").dt.tz_localize(None).dt.normalize(),
    "heure_src": pd.to_numeric(msg[C_HEURE], errors="coerce"),
    "p_pos": pd.to_numeric(msg[C_POS], errors="coerce"),
    "p_neg": pd.to_numeric(msg[C_NEG], errors="coerce"),
})
m["score"] = m["p_pos"] - m["p_neg"]

# Facebook s'appelait FB avant octobre 2021 ; les prix Yahoo sont sous META.
# Sans ce renommage, les messages $FB ne se raccrocheraient à aucun prix.
m["Ticker"] = m["Ticker"].replace({"FB": "META", "AAPL.": "AAPL"})
print("Tickers présents dans les messages :", sorted(m["Ticker"].unique()))

# --- Nettoyage ---------------------------------------------------------------
avant = len(m)
m = m.dropna(subset=["jour_src", "heure_src", "score", "Ticker"])
m = m[(m["heure_src"] >= 0) & (m["heure_src"] < 24)]
m = m[m["Ticker"].isin(TICKERS)]

# --- Horodatage reconstitué --------------------------------------------------
# Arrondi à la seconde : sans cela, 0.283333 h donne 19:16:59.999999998 et
# un message peut basculer d'une minute (voire d'une fenêtre) à l'autre.
m["ts_src"] = m["jour_src"] + pd.to_timedelta(np.round(m["heure_src"] * 3600), unit="s")
m["idx_src"] = m.index                      # lien vers la ligne d'origine de `msg`
m = m.sort_values("ts_src").reset_index(drop=True)

print(f"\n{avant:,} messages -> {len(m):,} retenus "
      f"({avant - len(m):,} écartés : date/heure/score invalide ou ticker hors périmètre)")
print(f"Période : {m['ts_src'].min()} -> {m['ts_src'].max()}\n")
print("Messages par ticker :")
print(m["Ticker"].value_counts().to_string())
print(f"\nScore : min {m['score'].min():+.3f} | médiane {m['score'].median():+.3f} "
      f"| moyenne {m['score'].mean():+.3f} | max {m['score'].max():+.3f}")

if C_NEU in msg.columns:
    neu = pd.to_numeric(msg[C_NEU], errors="coerce")
    print(f"\nMasse 'neutre' moyenne de FinBERT : {neu.mean():.3f}")
    print(f"Messages classés neutres à plus de 90 % : {(neu > 0.9).mean():.1%}")
    print("""
LECTURE : FinBERT a été entraîné sur de la PRESSE FINANCIÈRE (dépêches Reuters,
notes d'analystes). Il classe donc comme « neutre » une grande partie du jargon
de forum : emojis, « to the moon », listes de cashtags, salutations.

Ce n'est pas un bug, c'est un DÉCALAGE DE DOMAINE, et c'est une limite à
annoncer soi-même dans le mémoire. C'est aussi pour cela que le §1bis
ci-dessous vaut la peine d'être lu.
""")

---
# §1bis — Les étiquettes Bullish / Bearish : l'occasion à ne pas manquer

La description Kaggle du jeu de données précise :

> *« The dataset comes with "Bullish" and "Bearish" labelled by the user when sending out a message, which
> can be used for training a sentiment classifier. »*

Autrement dit, **une partie des messages porte une étiquette posée par son auteur lui-même**. Sur
StockTwits, l'utilisateur peut cocher « Bullish » ou « Bearish » en publiant. Environ 20 à 25 % des
messages sont ainsi étiquetés.

Ton fichier actuel ne conserve pas cette colonne — elle a disparu au nettoyage. **Il vaut la peine d'aller
la rechercher dans les fichiers Kaggle d'origine**, pour deux raisons.

### Raison 1 — valider FinBERT (le plus important)

Tu as un problème dont tu ne peux pas te débarrasser par le raisonnement : FinBERT met ~76 % de sa masse
sur « neutre » sur ton corpus. Un membre du jury demandera : *« votre mesure de sentiment mesure-t-elle
vraiment le sentiment ? »*

Les étiquettes utilisateur sont une **vérité terrain gratuite** pour y répondre. Sur les messages
étiquetés, tu peux mesurer :

- le **taux d'accord** entre le signe de FinBERT et l'étiquette de l'auteur ;
- l'**AUC** de FinBERT pour distinguer Bullish de Bearish.

Si l'accord est bon (disons AUC > 0,70), tu écris : *« sur les N messages portant une étiquette
utilisateur, FinBERT retrouve l'orientation déclarée avec une AUC de X ; l'instrument de mesure est donc
validé sur données externes. »* C'est un argument très solide, et **c'est une section de mémoire à lui
seul** — de la validation d'instrument, pas de la data science.

S'il est mauvais, c'est aussi un résultat : il faut alors utiliser les étiquettes plutôt que FinBERT.

### Raison 2 — une mesure de sentiment alternative, gratuite

Tu peux construire une seconde série de sentiment, sans aucun modèle :

```
bull_rate = (nombre de messages Bullish) / (Bullish + Bearish)   par fenêtre
```

Puis refaire toute la partie 4 avec elle. Si les deux mesures donnent la **même conclusion** — le
sentiment nocturne prédit le gap, pas la séance — ton résultat ne dépend plus du choix de FinBERT.
C'est le **test de robustesse le plus convaincant** que tu puisses offrir, et il coûte très peu.

> **Concrètement :** retourne aux CSV Kaggle, garde la colonne d'étiquette (souvent nommée `sentiment`,
> `label` ou `Bullish/Bearish`), et rejoins-la à ton fichier scoré via le texte brut ou l'identifiant du
> message. La cellule ci-dessous fait l'analyse dès que la colonne est présente ; sinon elle passe son
> tour sans bloquer.

In [ ]:
# Cette cellule s'exécute seulement si une colonne d'étiquette existe.
CANDIDATS_LABEL = ["sentiment", "label", "Sentiment", "Label", "bullish",
                   "user_sentiment", "Bullish_Bearish", "basic_sentiment"]
col_label = next((c_ for c_ in CANDIDATS_LABEL if c_ in msg.columns), None)

if col_label is None:
    print("Aucune colonne d'étiquette Bullish/Bearish dans ce fichier.")
    print("-> Section ignorée. Voir le texte ci-dessus pour la récupérer : c'est")
    print("   le test de robustesse le plus rentable du mémoire.")
else:
    from sklearn.metrics import roc_auc_score

    # idx_src relie chaque ligne de `m` à sa ligne d'origine dans `msg`
    lab = msg[col_label].reindex(m["idx_src"]).astype(str).str.lower().str.strip().values
    est_bull = pd.Series(lab, index=m.index).str.contains("bull", na=False)
    est_bear = pd.Series(lab, index=m.index).str.contains("bear", na=False)
    etiquetes = est_bull | est_bear

    print(f"Messages étiquetés par leur auteur : {etiquetes.sum():,} "
          f"({etiquetes.mean():.1%} du corpus)")
    print(f"  Bullish : {est_bull.sum():,}   |   Bearish : {est_bear.sum():,}")

    sub = m.loc[etiquetes].copy()
    sub["y_user"] = est_bull.loc[etiquetes].astype(int).values

    auc = roc_auc_score(sub["y_user"], sub["score"])
    accord = ((sub["score"] > 0) == (sub["y_user"] == 1)).mean()
    mb = sub.loc[sub["y_user"] == 1, "score"].mean()
    mo = sub.loc[sub["y_user"] == 0, "score"].mean()
    print(f"""
VALIDATION DE FinBERT SUR LES ÉTIQUETTES UTILISATEUR
  AUC (score FinBERT -> Bullish déclaré) : {auc:.4f}
  Taux d'accord sur le signe             : {accord:.1%}

  0.50 = FinBERT n'a aucun rapport avec ce que l'auteur déclare
  0.70 = accord correct, l'instrument est validé
  0.85 = très bon accord

  Score FinBERT moyen — messages Bullish : {mb:+.4f}
                        messages Bearish : {mo:+.4f}

Si l'AUC dépasse 0.70, tu peux écrire dans le mémoire : « sur les
{len(sub):,} messages portant une étiquette posée par leur auteur, FinBERT
retrouve l'orientation déclarée avec une AUC de {auc:.2f} ; l'instrument de
mesure est donc validé sur une source externe et indépendante. »
""")
    m["y_user"] = np.nan
    m.loc[etiquetes, "y_user"] = sub["y_user"].values

---
# §2 — Heure de New York ou UTC ? La question à trancher

**C'est la décision la plus lourde de conséquences de tout le notebook**, et ton fichier ne contient pas
la réponse : `Heure_decimale` est un nombre, sans fuseau attaché.

### Pourquoi c'est critique

StockTwits horodate ses messages en **UTC**. Si la personne qui a préparé le jeu de données a simplement
extrait l'heure sans convertir, alors `Heure_decimale` est en UTC. Or la bourse ouvre à **9h30 heure de
New York**, soit :

| Période | Fuseau de New York | 9h30 à New York = |
|---|---|---|
| novembre → mars | EST (UTC−5) | **14h30 UTC** |
| mars → novembre | EDT (UTC−4) | **13h30 UTC** |

Si l'on se trompe, **tous les messages changent de fenêtre**. Ceux de 14h UTC — c'est-à-dire juste avant
l'ouverture, donc de l'information **prédictive** — seraient rangés en `mkt`, une fenêtre
**contemporaine**. Le signal disparaîtrait, ou pire, une fuite d'information apparaîtrait.

### Comment trancher, sans avoir besoin de la documentation

Les marchés produisent une **empreinte horaire** très reconnaissable : l'activité explose à l'ouverture,
reste élevée pendant la séance, s'effondre la nuit. Il suffit donc de regarder **à quelle heure les gens
écrivent le plus** :

| Si le pic d'activité tombe entre… | alors `Heure_decimale` est en… |
|---|---|
| **9h30 et 16h00** | heure de New York — rien à convertir |
| **13h30 et 21h00** | UTC — il faut convertir |

Ce sont deux plages nettement différentes : le diagnostic est sans ambiguïté sur des millions de messages.

In [ ]:
# --- Le diagnostic : où se situe le pic d'activité ? ------------------------
profil = m.groupby(m["heure_src"].astype(int)).size()
profil = profil.reindex(range(24), fill_value=0)

part_ny  = profil.loc[9:15].sum() / profil.sum()     # séance si les heures sont en NY
part_utc = profil.loc[13:20].sum() / profil.sum()    # séance si les heures sont en UTC

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(profil.index, profil.values / 1000, color="#9aa5ae", width=.82)
ax.bar([h for h in range(9, 16)], [profil[h] / 1000 for h in range(9, 16)],
       color="#16324a", width=.82, label=f"séance si NY  ({part_ny:.0%} des messages)")
ax.bar([h for h in range(13, 21)], [profil[h] / 1000 for h in range(13, 21)],
       color="#c2703d", width=.45, label=f"séance si UTC ({part_utc:.0%} des messages)")
ax.set_xticks(range(24))
ax.set_xlabel("Heure_decimale (heure entière)")
ax.set_ylabel("milliers de messages")
ax.set_title("Dans quel fuseau les heures sont-elles exprimées ?",
             fontweight="bold", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(DOCS, "03_fig_diagnostic_fuseau.png"), dpi=150, bbox_inches="tight")
plt.show()

pic = int(profil.idxmax())
print(f"Heure la plus active : {pic}h   |   part 9h-16h : {part_ny:.1%}   "
      f"|   part 13h-21h : {part_utc:.1%}\n")

if part_ny > part_utc + 0.05:
    TZ_SOURCE = TZ_MARCHE
    print("=> VERDICT : les heures sont déjà en HEURE DE NEW YORK. Aucune conversion.")
elif part_utc > part_ny + 0.05:
    TZ_SOURCE = "UTC"
    print("=> VERDICT : les heures sont en UTC. Conversion vers l'heure de New York.")
else:
    TZ_SOURCE = "UTC"
    print("=> VERDICT INCERTAIN : les deux plages se valent. On suppose UTC (comportement")
    print("   par défaut de l'API StockTwits), mais VÉRIFIE avec la cellule suivante.")

print(f"\nTZ_SOURCE = {TZ_SOURCE!r}")
print("""
POUR LE MÉMOIRE : cette figure est un bon élément d'annexe. Elle montre que le
fuseau n'a pas été supposé mais ÉTABLI à partir des données, ce qui coupe court
à l'objection « comment savez-vous que vos fenêtres sont bien alignées ? ».
""")

### Un second contrôle, indépendant du premier

Le profil horaire peut être ambigu si la communauté est très internationale. On croise donc avec un
deuxième indice, plus fin : **le creux du week-end**.

L'activité chute le samedi et le dimanche. Mais le passage vendredi → samedi ne se fait pas à la même
heure selon le fuseau : si les heures sont en UTC, la baisse d'activité du « vendredi soir new-yorkais »
apparaît le **samedi matin UTC** (car 21h00 vendredi à New York = 02h00 samedi UTC). Une différence
d'activité entre le début et la fin de la journée du samedi trahit donc un décalage.

In [ ]:
sam = m[m["jour_src"].dt.dayofweek == 5]
ven = m[m["jour_src"].dt.dayofweek == 4]

if len(sam) and len(ven):
    tot_sam = len(sam) / max(sam["jour_src"].nunique(), 1)
    tot_ven = len(ven) / max(ven["jour_src"].nunique(), 1)
    print(f"Messages par jour — vendredi : {tot_ven:,.0f}   samedi : {tot_sam:,.0f}"
          f"   (rapport {tot_sam/tot_ven:.2f})")

    debut_sam = (sam["heure_src"] < 6).mean()
    print(f"Part des messages du samedi postés avant 6h : {debut_sam:.1%}")
    print("""
LECTURE : si les heures sont en UTC, le tout début du samedi UTC correspond au
vendredi soir new-yorkais -- encore actif. On attend donc une part NOTABLE de
messages avant 6h le samedi (typiquement > 25 %).
Si les heures sont déjà en heure de New York, le samedi avant 6h est une plage
morte, et cette part doit être FAIBLE (typiquement < 12 %).
""")
else:
    print("Pas assez de messages de week-end pour ce contrôle.")

# --- Conversion effective ---------------------------------------------------
if TZ_SOURCE == TZ_MARCHE:
    m["ts_ny"] = m["ts_src"]
    print("Heures conservées telles quelles (déjà en heure de marché).")
else:
    m["ts_ny"] = (m["ts_src"]
                  .dt.tz_localize(TZ_SOURCE, ambiguous="NaT", nonexistent="NaT")
                  .dt.tz_convert(TZ_MARCHE)
                  .dt.tz_localize(None))
    perdus = m["ts_ny"].isna().sum()
    m = m.dropna(subset=["ts_ny"])
    print(f"Converti de {TZ_SOURCE} vers {TZ_MARCHE}"
          f"{f' ({perdus} horodatages ambigus écartés)' if perdus else ''}.")
    print("""
POURQUOI tz_localize PUIS tz_convert, ET JAMAIS UNE SOUSTRACTION D'HEURES :
le décalage vaut -5 h en hiver et -4 h en été. Retrancher 5 h toute l'année
fausserait ~8 mois sur 12, en déplaçant les messages de fin de matinée d'une
fenêtre à l'autre. La bibliothèque connaît les dates de changement d'heure ;
nous, non.
""")

# --- Les deux colonnes dont le reste du notebook a besoin -------------------
m["date_ny"] = m["ts_ny"].dt.normalize()
m["heure"] = m["ts_ny"].dt.hour + m["ts_ny"].dt.minute / 60

print(f"\nAperçu du résultat :")
print(m[["Ticker", "jour_src", "heure_src", "ts_ny", "date_ny", "heure", "score"]].head(4).to_string())

---
# §3 — Le calendrier de bourse

Il faut savoir quels jours la bourse est ouverte. On pourrait installer une bibliothèque de calendriers
boursiers, mais il existe une source plus simple et **plus fiable** : **les prix eux-mêmes**.

Si Yahoo Finance renvoie une ligne pour une date, c'est que la bourse était ouverte ce jour-là. Ce
calendrier gère donc automatiquement :

- les week-ends,
- les jours fériés américains (Thanksgiving, Independence Day, Martin Luther King Day…),
- les fermetures exceptionnelles,
- les demi-séances (le prix existe, la logique reste valable).

**C'est un principe général qui vaut la peine d'être retenu** : quand deux sources doivent être alignées,
mieux vaut déduire le calendrier de la source contraignante que d'en importer un de l'extérieur, au risque
qu'il diverge.

In [ ]:
fin = pd.read_csv(FINANCE)
fin["Date"] = pd.to_datetime(fin["Date"]).dt.tz_localize(None).dt.normalize()
fin["Ticker"] = fin["Ticker"].astype(str).str.upper()
fin = fin[fin["Ticker"].isin(TICKERS)].sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"{len(fin):,} lignes de prix | {fin['Date'].min().date()} -> {fin['Date'].max().date()}")
print(fin.groupby("Ticker")["Date"].agg(["count", "min", "max"]).to_string())

# Le calendrier = les dates où AU MOINS un ticker a coté
CALENDRIER = np.array(sorted(fin["Date"].unique()), dtype="datetime64[ns]")
print(f"\n{len(CALENDRIER)} jours de bourse dans le calendrier")

# Contrôle : aucun week-end ne doit s'y trouver
jours_sem = pd.Series(CALENDRIER).dt.dayofweek
print(f"Week-ends présents (doit être 0) : {(jours_sem >= 5).sum()}")

# Quelques fériés bien connus, pour montrer qu'ils sont bien absents
for f in ["2020-01-20", "2020-07-03", "2020-11-26", "2020-12-25", "2021-07-05"]:
    present = np.datetime64(f) in CALENDRIER
    print(f"  {f} : {'PRÉSENT (anormal)' if present else 'absent — férié bien exclu'}")

> **Note importante sur la période couverte par les prix.** Le fichier de prix doit démarrer
> **avant** le 2 janvier 2020 — typiquement en novembre 2019. Deux raisons :
>
> 1. `prev_close` du 2 janvier 2020 est la clôture du 31 décembre 2019 : sans elle, le premier `gap`
>    serait vide.
> 2. `vol_20d` a besoin de 20 jours d'historique : sans amorçage, le premier mois serait vide.
>
> Les **messages**, eux, ne remontent pas avant 2020. C'est pourquoi les variables construites sur une
> fenêtre glissante de sentiment (`mu_night_z20`) sont vides au démarrage, alors que `vol_20d` ne l'est
> pas. Cette asymétrie est normale, et il faut savoir l'expliquer.

---
# §4 — Affecter chaque message à un jour de bourse et à une fenêtre

C'est **le cœur du notebook**. Tout le reste n'est que de l'agrégation.

### Les cinq fenêtres, définies sans ambiguïté

| Fenêtre | De | À |
|---|---|---|
| `overnight(D)` | **16h00 du jour de bourse précédent** | 00h00 du jour D |
| `pre(D)` | 00h00 du jour D | 09h30 du jour D |
| `night_full(D)` | = `overnight(D)` ∪ `pre(D)` | toute la période de fermeture |
| `open30(D)` | 09h30 | 10h00 |
| `mkt(D)` | 10h00 | 16h00 |
| `post(D)` | 16h00 du jour D | 00h00 du lendemain **calendaire** |

### La conséquence la plus importante : le lundi

Regarde bien la définition de `overnight` : elle part du **jour de bourse précédent**, pas du jour
calendaire précédent.

- **Mardi** → `overnight(mardi)` = lundi 16h00 → mardi 00h00. C'est exactement `post(lundi)`.
- **Lundi** → `overnight(lundi)` = **vendredi** 16h00 → lundi 00h00. Cela comprend le vendredi soir,
  **tout le samedi et tout le dimanche**.

D'où le fait vérifiable dans les données : `overnight(D) = post(D−1)` sur les jours consécutifs, mais
`overnight(lundi)` est bien plus gros que `post(vendredi)`.

### Deux affectations, pas une

Un message peut appartenir à **deux fenêtres à la fois** — et ce n'est pas un bug :

- un message du **jeudi 18h00** est dans `post(jeudi)` *et* dans `overnight(vendredi)` ;
- un message du **samedi 11h00** est dans `overnight(lundi)` seulement (le samedi n'est pas un jour de
  bourse, il n'a donc pas de fenêtre `post`).

Ce sont deux points de vue différents sur le même message : « ce qui s'est dit après la clôture de jeudi »
et « ce qui s'est dit avant l'ouverture de vendredi ». Les deux sont utiles, et ils vivent dans des
colonnes différentes.

On calcule donc **deux affectations** :

| Affectation | Fenêtres concernées | Jour de rattachement |
|---|---|---|
| **A — intra-journalière** | `pre`, `open30`, `mkt`, `post` | le jour calendaire du message, **s'il est un jour de bourse** |
| **B — nocturne** | `overnight`, `pre` | le **prochain jour de bourse** dont l'ouverture suit le message |

In [ ]:
# ============================================================================
#  AFFECTATION B — nocturne : à quel jour de bourse ce message se rapporte-t-il ?
# ============================================================================
d = m["date_ny"].values.astype("datetime64[ns]")
apres_cloture = (m["heure"] >= 16.0).values

# searchsorted 'left'  : premier jour de bourse >= date du message
# searchsorted 'right' : premier jour de bourse >  date du message
i_gauche = np.searchsorted(CALENDRIER, d, side="left")
i_droite = np.searchsorted(CALENDRIER, d, side="right")

# Un message posté APRÈS 16h se rattache forcément au jour de bourse SUIVANT.
# Un message posté avant 16h se rattache au premier jour de bourse >= sa date
# (donc à lui-même si c'est un jour ouvré, sinon au prochain : samedi -> lundi).
i_nuit = np.where(apres_cloture, i_droite, i_gauche)
valide = i_nuit < len(CALENDRIER)

m["jour_nuit"] = pd.NaT
m.loc[valide, "jour_nuit"] = CALENDRIER[i_nuit[valide]]
m["jour_nuit"] = pd.to_datetime(m["jour_nuit"])

# Dans la fenêtre nocturne : avant minuit -> 'overnight', après minuit -> 'pre'
avant_minuit = m["date_ny"] < m["jour_nuit"]
dans_pre = (m["date_ny"] == m["jour_nuit"]) & (m["heure"] < 9.5)

m["fen_nuit"] = np.where(avant_minuit, "overnight",
                  np.where(dans_pre, "pre", None))

# ============================================================================
#  AFFECTATION A — intra-journalière (uniquement les jours de bourse)
# ============================================================================
est_jour_bourse = np.isin(d, CALENDRIER)
h = m["heure"].values
m["fen_jour"] = np.where(~est_jour_bourse, None,
                  np.where(h < 9.5,  "pre",
                  np.where(h < 10.0, "open30",
                  np.where(h < 16.0, "mkt", "post"))))

print("RÉPARTITION DES MESSAGES\n" + "-" * 52)
print("Affectation A — intra-journalière :")
print(m["fen_jour"].value_counts(dropna=False).to_string())
print("\nAffectation B — nocturne :")
print(m["fen_nuit"].value_counts(dropna=False).to_string())
print(f"""
Messages hors période (postés après le dernier jour de bourse) : {(~valide).sum():,}
Messages un jour non ouvré (week-end / férié) : {(~est_jour_bourse).sum():,}
  -> ils n'ont PAS de fenêtre intra-journalière, mais ils comptent
     bien dans la fenêtre nocturne du prochain jour de bourse.
""")

### Vérifier l'affectation avant d'aller plus loin

Trois contrôles. Si l'un échoue, inutile de continuer : le panel serait faux.

In [ ]:
print("CONTRÔLE 1 — un lundi absorbe-t-il bien le week-end ?")
print("-" * 62)
lundis = m[(m["jour_nuit"].dt.dayofweek == 0) & (m["fen_nuit"] == "overnight")]
autres = m[(m["jour_nuit"].dt.dayofweek.isin([1, 2, 3, 4])) & (m["fen_nuit"] == "overnight")]
n_lundi = lundis.groupby(["Ticker", "jour_nuit"]).size().mean()
n_autre = autres.groupby(["Ticker", "jour_nuit"]).size().mean()
print(f"  messages overnight, en moyenne — lundi : {n_lundi:8.0f}")
print(f"                                   autres : {n_autre:8.0f}")
print(f"  rapport : {n_lundi / n_autre:.2f}x")
print("  -> doit être NETTEMENT supérieur à 1 (le lundi couvre ~3 jours civils).\n")

print("CONTRÔLE 2 — aucun message de séance ne doit tomber en fenêtre nocturne")
print("-" * 62)
fuite = m[(m["fen_nuit"].notna()) & (m["heure"] >= 9.5) & (m["heure"] < 16.0)
          & (m["date_ny"] == m["jour_nuit"])]
print(f"  messages en infraction : {len(fuite):,}   -> doit valoir 0\n")

print("CONTRÔLE 3 — night_full = overnight + pre, message par message")
print("-" * 62)
a = (m["fen_nuit"] == "overnight").sum()
b = (m["fen_nuit"] == "pre").sum()
t = m["fen_nuit"].notna().sum()
print(f"  overnight {a:,} + pre {b:,} = {a + b:,}   |   night_full {t:,}")
print(f"  cohérent : {a + b == t}")

---
# §5 — Agréger : les sept statistiques

Pour chaque triplet (**action**, **jour de bourse**, **fenêtre**), on résume tous les messages en sept
nombres.

| Statistique | Calcul | Ce qu'elle capte |
|---|---|---|
| `n` | nombre de messages | **volume d'attention** |
| `mu` | moyenne des `score` | **direction** de l'opinion |
| `sd` | écart-type des `score` | **désaccord** entre intervenants |
| `pos` | moyenne de `p_pos` | intensité de l'optimisme |
| `neg` | moyenne de `p_neg` | intensité du pessimisme |
| `p10` | 1<sup>er</sup> décile des `score` | le versant pessimiste |
| `p90` | 9<sup>e</sup> décile des `score` | le versant optimiste |

Deux remarques qui comptent pour la soutenance.

**1. `mu = pos − neg` est une identité, pas une coïncidence.** Puisque `score = p_pos − p_neg` pour chaque
message, la moyenne des scores est la différence des moyennes. Conséquence : ces trois variables ne peuvent
jamais figurer ensemble dans une régression (colinéarité parfaite).

**2. On calcule `sd`, `p10` et `p90` de `night_full` directement sur les messages**, et non en combinant
`overnight` et `pre`. C'est indispensable, et c'est ce qui corrige deux défauts du panel actuel — voir §12.

In [ ]:
def agreger(sous_ensemble, cle_jour, nom_fenetre):
    """Résume les messages d'une fenêtre en 7 statistiques par (Ticker, jour)."""
    if len(sous_ensemble) == 0:
        return pd.DataFrame()
    g = (sous_ensemble.groupby(["Ticker", cle_jour])
         .agg(n=("score", "size"),
              mu=("score", "mean"),
              sd=("score", "std"),
              p10=("score", lambda s: s.quantile(0.10)),
              p90=("score", lambda s: s.quantile(0.90)),
              pos=("p_pos", "mean"),
              neg=("p_neg", "mean"))
         .reset_index()
         .rename(columns={cle_jour: "Date"}))
    g.columns = ["Ticker", "Date"] + [f"{c_}_{nom_fenetre}" for c_ in
                                      ["n", "mu", "sd", "p10", "p90", "pos", "neg"]]
    return g

blocs = {}

# --- fenêtres nocturnes (clé = jour_nuit) -----------------------------------
blocs["overnight"]  = agreger(m[m["fen_nuit"] == "overnight"], "jour_nuit", "overnight")
blocs["pre"]        = agreger(m[m["fen_nuit"] == "pre"],       "jour_nuit", "pre")
# night_full : calculé sur l'UNION des messages, pas par recombinaison
blocs["night_full"] = agreger(m[m["fen_nuit"].notna()],        "jour_nuit", "night_full")

# --- fenêtres intra-journalières (clé = date_ny) ----------------------------
for f in ["open30", "mkt", "post"]:
    blocs[f] = agreger(m[m["fen_jour"] == f], "date_ny", f)

for nom, b in blocs.items():
    print(f"  {nom:12s} : {len(b):5,} lignes (action × jour)")

print("""
POURQUOI night_full EST CALCULÉ SUR L'UNION DES MESSAGES

Les moyennes s'additionnent, les écarts-types et les percentiles NON.
  - la moyenne d'une union se déduit des moyennes pondérées : OK
  - l'écart-type d'une union ne se déduit PAS des écarts-types :
    il faut aussi la variance ENTRE les deux groupes
  - le 9e décile d'une union ne se déduit pas des 9es déciles

En repartant des messages, ces trois statistiques sont exactes par
construction, et disp_night_full (= p90 - p10) devient calculable.
""")

---
# §6 — Passer au format large

Les six blocs ont chacun une ligne par (action, jour). On les fusionne pour obtenir **une seule ligne par
action-jour**, avec toutes les fenêtres côte à côte.

On part du **squelette** — le produit cartésien des actions et des jours de bourse — plutôt que des
messages. C'est important : ainsi, un jour où personne n'a parlé d'une action existe quand même dans le
panel, avec `n = 0`. Si on partait des messages, ce jour disparaîtrait silencieusement et le panel serait
déséquilibré.

In [ ]:
# Squelette : toutes les combinaisons action × jour de bourse de la période
cal = pd.Series(CALENDRIER, name="Date")
cal = cal[(cal >= DEBUT) & (cal <= FIN)]
panel = (pd.MultiIndex.from_product([TICKERS, cal], names=["Ticker", "Date"])
           .to_frame(index=False))
print(f"Squelette : {len(panel):,} lignes ({len(TICKERS)} actions × {len(cal)} jours)")

for nom, b in blocs.items():
    if len(b):
        panel = panel.merge(b, on=["Ticker", "Date"], how="left")

# Un jour sans message n'est pas une donnée manquante : c'est un comptage de 0
for f in ["overnight", "pre", "night_full", "open30", "mkt", "post"]:
    if f"n_{f}" in panel.columns:
        panel[f"n_{f}"] = panel[f"n_{f}"].fillna(0.0)

print(f"Après fusion : {panel.shape[0]:,} lignes x {panel.shape[1]} colonnes")
print("\nJours sans aucun message nocturne, par ticker :")
print(panel.assign(muet=panel["n_night_full"] == 0).groupby("Ticker")["muet"].sum().to_string())
print("""
NOTE : un écart-type demande au moins DEUX messages, un décile en demande
plusieurs. Les nuits à 0 ou 1 message auront donc n = 0 ou 1 mais sd = NaN.
C'est le comportement correct : on ne fabrique pas une dispersion à partir
d'une seule observation.
""")

---
# §7 — Les variables de marché

Sept colonnes, toutes calculées **à l'intérieur de chaque ticker** (`groupby("Ticker")`). C'est un détail
qui n'en est pas un : sans le `groupby`, la clôture de la veille d'AAPL au premier jour serait la dernière
clôture d'AMZN, et le premier `gap` de chaque ticker serait absurde.

In [ ]:
fin = fin.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Ton fichier de collecte contient DÉJÀ prev_close, gap, ret_oc et ret_cc.
# On les recalcule quand même, puis on compare : si les deux versions
# coïncident, cela valide d'un coup le script de collecte ET ce notebook.
deja = [c_ for c_ in ["prev_close", "gap", "ret_oc", "ret_cc"] if c_ in fin.columns]
if deja:
    ancien = fin[deja].copy()
    print(f"Colonnes déjà présentes dans le fichier de collecte : {deja}")
    print("-> recalculées ici, puis comparées.\n")

g = fin.groupby("Ticker")

fin["prev_close"]  = g["Close"].shift(1)
fin["prev_volume"] = g["Volume"].shift(1)

fin["gap"]    = fin["Open"] / fin["prev_close"] - 1     # 16h (J-1)  -> 9h30 (J)
fin["ret_oc"] = fin["Close"] / fin["Open"] - 1          # 9h30 (J)   -> 16h  (J)
fin["ret_cc"] = fin["Close"] / fin["prev_close"] - 1    # 16h (J-1)  -> 16h  (J)

fin["prev_ret_cc"] = fin.groupby("Ticker")["ret_cc"].shift(1)

# Volatilité réalisée sur 20 jours : le .shift(1) exclut le jour J lui-même
fin["vol_20d"] = fin.groupby("Ticker")["ret_cc"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=10).std())

if deja:
    print("RECOUPEMENT avec les colonnes du fichier de collecte")
    print("-" * 58)
    for c_ in deja:
        ecart = (fin[c_] - ancien[c_]).abs().max()
        verdict = "identique" if (pd.isna(ecart) or ecart < 1e-9) else "DIVERGENCE"
        print(f"  {c_:12s} : écart max = {ecart:.2e}   -> {verdict}")
    print("""
Un écart non nul signalerait que le script de collecte et ce notebook ne
définissent pas la même chose -- typiquement un groupby oublié, ou un tri
différent. « identique » partout = les deux chaînes sont cohérentes.
""")

print("Erreur maximale sur l'identité de décomposition :")
ctrl = fin.dropna(subset=["gap", "ret_oc", "ret_cc"])
err = ((1 + ctrl["gap"]) * (1 + ctrl["ret_oc"]) - (1 + ctrl["ret_cc"])).abs().max()
print(f"  |(1+gap)(1+ret_oc) - (1+ret_cc)| = {err:.2e}   -> doit être < 1e-12\n")

r = fin.iloc[0]
print(f"""EXEMPLE — {r['Ticker']} le {r['Date'].date()}
  clôture veille {r['prev_close']:.4f} -> ouverture {r['Open']:.4f}  : gap    = {r['gap']:+.6f}
  ouverture {r['Open']:.4f} -> clôture {r['Close']:.4f}              : ret_oc = {r['ret_oc']:+.6f}
  composition : (1{r['gap']:+.6f}) x (1{r['ret_oc']:+.6f}) - 1       = {(1+r['gap'])*(1+r['ret_oc'])-1:+.6f}
""")
print(fin.groupby("Ticker")[["gap", "ret_oc", "ret_cc", "vol_20d"]].mean().mul(100).round(4).to_string())

---
# §8 — Fusionner texte et marché

Une jointure à gauche sur (`Ticker`, `Date`), en partant du **marché**. On ne garde donc que les jours où
la bourse était ouverte — ce qui est cohérent avec le fait que les cibles n'existent que ces jours-là.

In [ ]:
COLS_FIN = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume",
            "prev_close", "gap", "ret_oc", "ret_cc", "prev_ret_cc", "prev_volume", "vol_20d"]

pan = (fin[COLS_FIN]
       .merge(panel, on=["Ticker", "Date"], how="left")
       .query("@DEBUT <= Date <= @FIN")
       .sort_values(["Ticker", "Date"])
       .reset_index(drop=True))

print(f"Panel fusionné : {pan.shape[0]:,} lignes x {pan.shape[1]} colonnes")
print(f"Jours par ticker :\n{pan.groupby('Ticker')['Date'].nunique().to_string()}")
print(f"\nPanel équilibré : {pan.groupby('Ticker')['Date'].nunique().nunique() == 1}")
print("""
SI LE PANEL N'EST PAS ÉQUILIBRÉ : un ticker a moins de jours que les autres.
La cause la plus fréquente est un trou dans les données de PRIX, pas dans les
messages -- puisque la jointure part du marché. Vérifier le fichier finance.
""")

---
# §9 — Les variables d'attention

Trois transformations, appliquées aux fenêtres `night_full`, `open30`, `mkt` et `post`.

### `nlog` — pourquoi passer au logarithme

La distribution du nombre de messages est **très asymétrique** : la plupart des nuits sont calmes,
quelques-unes explosent (résultats trimestriels, tweet d'Elon Musk, squeeze). L'écart-type est du même
ordre que la moyenne.

Donné brut à un modèle linéaire, `n` serait dominé par une poignée de jours extrêmes. Le logarithme
comprime les valeurs hautes et rend la variable exploitable. Le `1 +` évite `ln(0)` les nuits muettes.

### `nabn` — le choc d'attention

C'est **la variable la plus intéressante du panel** pour un mémoire d'actuariat. Elle ne mesure pas
« combien on parle », mais **« combien on parle par rapport à d'habitude »**.

Une valeur de +1,5 dit : *« cette nuit, on parle beaucoup plus de ce titre que sur le mois écoulé »*.
La littérature montre qu'un choc d'attention annonce une journée agitée (Da, Engelberg & Gao 2011 ;
Antweiler & Frank 2004).

> **⚠ Le point à ne pas rater — et le bug du panel actuel.** La moyenne de référence doit :
>
> 1. être décalée d'un jour (`.shift(1)`) — sinon elle contient le jour qu'on veut comparer ;
> 2. diviser par le **nombre réel** d'observations disponibles (`min_periods`), et non par 20 en dur.
>
> Dans le panel actuel, le dénominateur est fixé à 20 même quand il n'y a que 3 jours d'historique.
> Résultat : la référence est quasi nulle au démarrage, et `nabn` vaut ~5,8 au lieu de ~0 pendant le
> premier mois de chaque titre. Cela concerne ~100 lignes sur 2 740. La version ci-dessous corrige cela
> en laissant `NaN` tant qu'il n'y a pas assez d'historique — un vide honnête vaut mieux qu'un chiffre faux.

### `disp` — le désaccord robuste

`disp = p90 − p10`, l'intervalle interdécile. Contrairement à l'écart-type, il ignore les quelques
messages extrêmes et décrit la largeur du cœur de la distribution.

In [ ]:
FENETRES_ATT = ["night_full", "open30", "mkt", "post"]
REF_JOURS, MIN_JOURS = 20, 10

for f in FENETRES_ATT:
    if f"n_{f}" not in pan.columns:
        continue
    pan[f"nlog_{f}"] = np.log1p(pan[f"n_{f}"])

    # Référence : moyenne des REF_JOURS jours PRÉCÉDENTS, avec un min_periods
    # (et non une division par REF_JOURS en dur -- voir l'encadré ci-dessus)
    pan[f"nabn_{f}"] = pan[f"nlog_{f}"] - pan.groupby("Ticker")[f"nlog_{f}"].transform(
        lambda s: s.shift(1).rolling(REF_JOURS, min_periods=MIN_JOURS).mean())

    if f"p90_{f}" in pan.columns:
        pan[f"disp_{f}"] = pan[f"p90_{f}"] - pan[f"p10_{f}"]

print("Statistiques des variables d'attention :")
cols = [c_ for c_ in pan.columns if c_.startswith(("nlog_", "nabn_", "disp_"))]
print(pan[cols].describe().T[["count", "mean", "std", "min", "max"]].round(3).to_string())

print(f"""
CONTRÔLE — nabn doit être CENTRÉ SUR ZÉRO
  moyenne de nabn_night_full : {pan['nabn_night_full'].mean():+.4f}   (attendu : ~0)
  valeurs manquantes         : {pan['nabn_night_full'].isna().sum()}  (les {MIN_JOURS} premiers jours de chaque titre)

Dans la version actuelle du panel, cette moyenne est fortement POSITIVE et
nabn vaut ~5.8 le premier jour. C'est la signature du bug de dénominateur.

CONTRÔLE — disp_night_full est maintenant renseigné
  valeurs manquantes : {pan['disp_night_full'].isna().sum()} sur {len(pan)}
  (contre 100 % dans le panel actuel : les déciles de night_full n'y étaient
   pas calculés, car on ne peut pas combiner les déciles de deux fenêtres.)
""")

---
# §10 — Les dérivées nocturnes

Trois façons différentes de répondre à la question **« ce sentiment est-il élevé ? »**. Elles portent
toutes sur `mu_night_full`, la variable centrale du mémoire.

| Variable | Question à laquelle elle répond |
|---|---|
| `dmu_night` | le sentiment a-t-il **changé** depuis hier ? |
| `mu_night_ma3` | quelle est la **tendance** des trois dernières nuits ? |
| `mu_night_z20` | ce niveau est-il **anormal pour ce titre** ? |

### Pourquoi `mu_night_z20` est la plus importante

Le sentiment moyen vaut **0,160 pour NVDA** et **0,044 pour TSLA** — un facteur 4. Ce n'est pas que la
foule aime 4 fois plus NVDA : c'est que les deux communautés **écrivent différemment**. Un sentiment de
0,12 est donc médiocre pour NVDA et exceptionnel pour TSLA.

Sans normalisation, un modèle nourri de valeurs brutes apprend surtout **l'identité du titre**. Avec le
z-score, `+2` signifie *« anormalement positif pour ce titre, comparé à son propre mois écoulé »* — et
c'est comparable d'un titre à l'autre.

> **Le `.shift(1)` est obligatoire.** Sans lui, la moyenne des 20 jours inclurait le jour J, et on
> normaliserait une valeur par une moyenne qui la contient. C'est une fuite discrète mais réelle, et c'est
> l'erreur la plus fréquente en finance quantitative appliquée.

In [ ]:
gt = pan.groupby("Ticker")["mu_night_full"]

# 1) variation d'un jour à l'autre
pan["dmu_night"] = gt.diff()

# 2) moyenne mobile 3 jours, jour courant INCLUS
#    -> légitime ici : mu_night_full(J) est connu à 9h30, avant le gap
pan["mu_night_ma3"] = gt.transform(lambda s: s.rolling(3, min_periods=2).mean())

# 3) z-score sur 20 jours, référence STRICTEMENT passée
def zscore_causal(s, fenetre=20, minp=20):
    mu = s.shift(1).rolling(fenetre, min_periods=minp).mean()
    sd = s.shift(1).rolling(fenetre, min_periods=minp).std()
    return (s - mu) / sd.replace(0, np.nan)

pan["mu_night_z20"] = gt.transform(zscore_causal)

print(pan[["dmu_night", "mu_night_ma3", "mu_night_z20"]].describe().T.round(4).to_string())

print(f"""
CONTRÔLE DE LA NORMALISATION

Moyenne de mu_night_full par ticker (AVANT normalisation) :
{pan.groupby('Ticker')['mu_night_full'].mean().round(4).to_string()}

Moyenne de mu_night_z20 par ticker (APRÈS normalisation) :
{pan.groupby('Ticker')['mu_night_z20'].mean().round(4).to_string()}

-> Les écarts de NIVEAU entre titres ont disparu : c'est l'effet recherché.
   Chaque titre est désormais comparé à sa propre histoire.
""")

---
# §11 — Les variables décalées et les cibles

### Les six colonnes `_lag1` : récupérer l'information interdite

Les fenêtres `mkt` et `post` sont **interdites** pour le jour J : elles se ferment après l'ouverture, donc
elles réagissent au prix au lieu de l'annoncer.

Mais celles du **jour J−1** sont parfaitement connues à 16h00 la veille — donc parfaitement légales. C'est
tout l'objet du décalage : **récupérer le climat de la séance précédente sans commettre de fuite**.

Elles servent aussi de **variable de contrôle** : si le sentiment nocturne garde son pouvoir prédictif une
fois le sentiment de la veille pris en compte, alors c'est bien l'information *nouvelle* de la nuit qui
compte, et non un simple prolongement de la journée passée.

### Les trois cibles binaires

`y_gap`, `y_oc`, `y_cc` valent 1 si le rendement correspondant est positif.

> **Le seuil à zéro est économiquement naïf.** Un gap de +0,013 % (1,3 point de base) donne `y_gap = 1`,
> alors que ce mouvement est plusieurs fois plus petit que les frais de transaction. On ajoute donc une
> quatrième cible, `y_gap_net`, qui **neutralise** les jours trop petits pour être tradés — c'est elle qui
> compte pour le backtest.

In [ ]:
# --- décalages des fenêtres non prédictives ---------------------------------
for f in ["mkt", "post"]:
    for stat in ["mu", "nlog", "sd"]:
        src = f"{stat}_{f}"
        if src in pan.columns:
            pan[f"{src}_lag1"] = pan.groupby("Ticker")[src].shift(1)

# On laisse NaN plutôt que 0 : au premier jour, l'information n'est pas
# "zéro message", elle est "inconnue". Un 0 sur une échelle logarithmique
# signifierait exactement 0 message, ce qui serait faux.

# --- cibles binaires ---------------------------------------------------------
for cible, source in [("y_gap", "gap"), ("y_oc", "ret_oc"), ("y_cc", "ret_cc")]:
    pan[cible] = (pan[source] > 0).astype("Int64")
    pan.loc[pan[source].isna(), cible] = pd.NA

# --- cible à zone morte ------------------------------------------------------
SEUIL_BP = 15                                    # 15 points de base = 0,15 %
seuil = SEUIL_BP / 10_000
pan["y_gap_net"] = np.where(pan["gap"] > seuil, 1,
                     np.where(pan["gap"] < -seuil, 0, np.nan))

print("Équilibre des classes (proportion de 1) :")
for c_ in ["y_gap", "y_oc", "y_cc", "y_gap_net"]:
    s = pan[c_].dropna()
    print(f"  {c_:11s} : {s.astype(float).mean():.3f}   (n = {len(s):,})")

neutres = int(pan["y_gap_net"].isna().sum() - pan["gap"].isna().sum())
print(f"""
Jours neutralisés par la zone morte de {SEUIL_BP} bp : {neutres:,} ({neutres/len(pan)*100:.1f} %)

ATTENTION AU TAUX DE BASE : y_gap tourne autour de 0.56, pas 0.50.
Un modèle qui répondrait « hausse » tous les jours obtiendrait donc 56 %
d'exactitude SANS RIEN APPRENDRE. C'est pourquoi la partie 6 reporte l'AUC
et le MCC, et n'utilise l'exactitude que comparée à ce taux de base.
""")

---
# §12 — Contrôles qualité et export

Sept contrôles automatiques. Si l'un échoue, le panel n'est pas exporté : mieux vaut pas de fichier qu'un
fichier faux.

In [ ]:
echecs = []
def verifier(nom, condition, detail=""):
    print(f"  [{'OK   ' if condition else 'ÉCHEC'}] {nom}" + (f"  — {detail}" if detail else ""))
    if not condition:
        echecs.append(nom)

print("CONTRÔLES QUALITÉ\n" + "=" * 70)

# 1. panel équilibré
n_j = pan.groupby("Ticker")["Date"].nunique()
verifier("Panel équilibré", n_j.nunique() == 1, f"{n_j.iloc[0]} jours par ticker")

# 2. pas de doublon
verifier("Aucun doublon (Ticker, Date)", not pan.duplicated(["Ticker", "Date"]).any())

# 3. identité de décomposition du rendement
ctrl = pan.dropna(subset=["gap", "ret_oc", "ret_cc"])
e = ((1 + ctrl["gap"]) * (1 + ctrl["ret_oc"]) - (1 + ctrl["ret_cc"])).abs().max()
verifier("(1+gap)(1+ret_oc) = 1+ret_cc", e < 1e-9, f"erreur max {e:.2e}")

# 4. identité mu = pos - neg
err_max = 0
for f in ["overnight", "pre", "night_full", "open30", "mkt", "post"]:
    s = pan.dropna(subset=[f"mu_{f}", f"pos_{f}", f"neg_{f}"])
    if len(s):
        err_max = max(err_max, (s[f"pos_{f}"] - s[f"neg_{f}"] - s[f"mu_{f}"]).abs().max())
verifier("mu = pos - neg sur toutes les fenêtres", err_max < 1e-9, f"erreur max {err_max:.2e}")

# 5. additivité des comptages
s = pan.dropna(subset=["n_overnight", "n_pre", "n_night_full"])
e = (s["n_overnight"] + s["n_pre"] - s["n_night_full"]).abs().max()
verifier("n_night_full = n_overnight + n_pre", e < 1e-9, f"erreur max {e:.2e}")

# 6. nabn centré
mn = pan["nabn_night_full"].mean()
verifier("nabn centré sur zéro", abs(mn) < 0.15, f"moyenne {mn:+.4f}")

# 7. disp_night_full renseigné
part = pan["disp_night_full"].notna().mean()
verifier("disp_night_full calculable", part > 0.9, f"{part:.1%} de lignes renseignées")

print("=" * 70)
print("TOUS LES CONTRÔLES PASSENT." if not echecs
      else "ÉCHECS :\n  - " + "\n  - ".join(echecs))

In [ ]:
ORDRE = (["Date", "Open", "High", "Low", "Close", "Volume", "Ticker", "prev_close",
          "gap", "ret_oc", "ret_cc", "prev_ret_cc", "prev_volume", "vol_20d"]
         + [f"{k}_{f}" for f in ["overnight", "pre", "night_full", "open30", "mkt", "post"]
            for k in ["n", "mu", "sd", "p10", "p90", "pos", "neg"]]
         + [f"{k}_{f}" for f in ["night_full", "open30", "mkt", "post"]
            for k in ["nlog", "nabn", "disp"]]
         + ["dmu_night", "mu_night_ma3", "mu_night_z20"]
         + [f"{k}_{f}_lag1" for f in ["mkt", "post"] for k in ["mu", "nlog", "sd"]]
         + ["y_gap", "y_oc", "y_cc", "y_gap_net"])
ORDRE = [c_ for c_ in ORDRE if c_ in pan.columns]
sortie = pan[ORDRE].copy()

if echecs:
    print("Export ANNULÉ : corrige les contrôles en échec d'abord.")
else:
    sortie.to_csv(SORTIE, index=False)
    print(f"Écrit : {SORTIE}")
    print(f"        {sortie.shape[0]:,} lignes x {sortie.shape[1]} colonnes")

print("\nAperçu des 3 premières lignes :")
print(sortie.head(3).iloc[:, :14].to_string())

---
# §13 — Ce qui change par rapport au panel actuel

Ce notebook reconstruit le panel avec trois corrections. Aucune ne renverse les conclusions de la partie 4,
mais toutes les trois sont le genre de détail qu'un jury attentif repère — et il vaut mieux les avoir
signalées soi-même.

### 1. `disp_night_full` devient calculable

**Avant :** colonne entièrement vide. `disp = p90 − p10`, or `p10_night_full` et `p90_night_full`
n'existaient pas — le script tentait de combiner les déciles de `overnight` et `pre`, ce qui est
**mathématiquement impossible** : les percentiles ne s'additionnent pas.

**Maintenant :** les déciles de `night_full` sont calculés directement sur l'union des messages. La colonne
est renseignée.

### 2. `nabn` n'est plus faussé au démarrage

**Avant :** la référence divisait par 20 même quand moins de 20 jours étaient disponibles. Le premier jour,
`nabn = nlog = 5,76` au lieu de ~0. Environ 100 lignes sur 2 740 concernées (5 titres × 20 jours), toutes
en janvier-février 2020.

**Maintenant :** `min_periods=10` — la référence divise par le nombre réel d'observations, et reste `NaN`
tant qu'il n'y a pas assez d'historique. Un vide honnête vaut mieux qu'un chiffre faux.

### 3. `sd_night_full` est un vrai écart-type

**Avant :** moyenne pondérée des écarts-types de `overnight` et `pre`. Cela **sous-estime** la dispersion
réelle, parce que cela ignore l'écart **entre** les moyennes des deux fenêtres.

La formule correcte de la variance d'une union comporte deux termes :

```
variance totale  =  variance INTRA  +  variance INTER
                    (dispersion       (écart entre les
                     dans chaque       moyennes des deux
                     fenêtre)          fenêtres)
```

Sur la première ligne d'AAPL, la valeur stockée est 0,4082 alors que le vrai écart-type poolé est 0,4098 —
un écart de 0,4 %, faible ici mais qui grandit quand les deux fenêtres divergent.

**Maintenant :** `sd_night_full` est calculé directement sur les messages de l'union. Le problème disparaît
par construction.

---

## Ce qu'il faut retenir de la partie 3

| Choix | Pourquoi il est critique |
|---|---|
| Conversion de fuseau par `tz_convert` | un décalage fixe fausse ~8 mois sur 12 à cause de l'heure d'été |
| Calendrier déduit **des prix** | gère week-ends, fériés et fermetures exceptionnelles sans bibliothèque externe |
| `overnight` part du **jour de bourse** précédent | c'est ce qui fait que la fenêtre du lundi absorbe tout le week-end |
| Deux affectations, pas une | un message du jeudi soir est à la fois `post(jeudi)` et `overnight(vendredi)` |
| Squelette = produit cartésien | un jour sans message existe quand même, avec `n = 0` : le panel reste équilibré |
| `sd`, `p10`, `p90` calculés sur l'union | les moyennes s'additionnent, la dispersion et les percentiles non |
| `.shift(1)` sur toute fenêtre glissante | sans lui, la référence contient le jour qu'on veut comparer |

**La phrase à savoir dire en soutenance :**

> « Le panel est construit en rattachant chaque message à un jour de bourse et à une phase de la journée,
> après conversion en heure de New York. Le calendrier est déduit des cotations elles-mêmes, ce qui gère
> les fériés. La fenêtre nocturne part de la clôture du jour de bourse précédent, si bien que celle du
> lundi couvre l'ensemble du week-end. Les statistiques de dispersion sont calculées directement sur les
> messages, et non recombinées à partir des sous-fenêtres, parce que ni les écarts-types ni les percentiles
> ne sont additifs. »